# Notebook 1: small end-to-end test of the whole pipeline

Neural Nexus, vector-DB compression project.

What this notebook does, in order:

1. Load 18,000 Natural Questions passages, cut each to its first 32 tokens.
2. Victim side: embed 10,000 passages with BGE-large (the victim database) and their 10,000 questions.
3. Attacker side: embed 8,000 different passages with BGE and GTR-base (the attacker's own public data).
4. Build the 6 compressed databases (fp32, int8, int4, PQ, binary, SVD-256) and run correctness tests on each.
5. Retrieval: Top-10 recall for each compressed database (how useful the DB still is).
6. Positive control: vec2text on true GTR vectors. Must give BLEU above 80, otherwise the notebook stops.
7. Alignment: a supervised linear map from BGE space to GTR space (ridge regression). This is an UPPER BOUND, not the unsupervised attack. The unsupervised attack (mini-vec2vec) is Notebook 2.
8. Inversion: vec2text on 100 aligned vectors per scheme, scored with BLEU and token F1.
9. Results table, plot, and files saved to /kaggle/working.

Settings before you press Run All:
- Accelerator: GPU T4 (x1 or x2, only one is used)
- Internet: ON
- Expected runtime: about 25-35 minutes.

In [ ]:
# Cell 1 - install. transformers is pinned because vec2text 0.0.13 breaks on newer versions
# (this exact setup worked on a Kaggle T4 on 23 Oct 2026).
!pip install -q "transformers==4.46.3" "vec2text==0.0.13" faiss-cpu sacrebleu

In [ ]:
# Cell 2 - settings. Change sizes here only.
import os, json, time, random
import numpy as np, torch

SEED = 0
N_VICTIM = 10_000      # victim database size
N_ATTACKER = 8_000     # attacker's own public passages (disjoint from the victim DB)
N_INVERT = 100         # victim passages we try to reconstruct per scheme
MAX_TOKENS = 32        # the pretrained inverter was trained on 32-token texts
INV_STEPS = 20         # 20 correction steps: BLEU 91.2 in our 23 Oct benchmark
INV_BATCH = 25
OUT = "/kaggle/working/nb1"
os.makedirs(OUT, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU. Turn on GPU T4 in the notebook settings."
print(torch.__version__, torch.cuda.get_device_name(0))
import transformers; print("transformers", transformers.__version__)

In [ ]:
# Cell 3 - data. Cut every passage to its first 32 GTR (T5) tokens,
# drop duplicates, then split into victim DB and attacker set (no overlap).
from datasets import load_dataset
from transformers import AutoTokenizer

gtr_tok = AutoTokenizer.from_pretrained("sentence-transformers/gtr-t5-base")
ds = load_dataset("sentence-transformers/natural-questions", split="train")
print(ds)

order = np.random.default_rng(SEED).permutation(len(ds))
need = N_VICTIM + N_ATTACKER
queries, passages, seen = [], [], set()
for start in range(0, len(order), 4000):
    idx = order[start:start + 4000].tolist()
    rows = ds.select(idx)
    enc = gtr_tok(list(rows["answer"]), max_length=MAX_TOKENS, truncation=True)
    cut = gtr_tok.batch_decode(enc["input_ids"], skip_special_tokens=True)
    for q, p in zip(list(rows["query"]), cut):
        p = p.strip()
        if p and p not in seen:
            seen.add(p); queries.append(q); passages.append(p)
    if len(passages) >= need:
        break
queries, passages = queries[:need], passages[:need]

victim_q, victim_p = queries[:N_VICTIM], passages[:N_VICTIM]
att_p = passages[N_VICTIM:need]
inv_idx = np.random.default_rng(SEED + 1).choice(N_VICTIM, N_INVERT, replace=False)
json.dump({"victim_q": victim_q, "victim_p": victim_p, "att_p": att_p,
           "inv_idx": inv_idx.tolist()}, open(f"{OUT}/texts.json", "w"))
print(len(victim_p), "victim passages,", len(att_p), "attacker passages")
print("example:", victim_p[0])

In [ ]:
# Cell 4 - embedding functions.
# BGE-large: CLS token, L2-normalised (how BGE is meant to be used). Questions get BGE's search prefix.
# GTR: mean-pooled T5 ENCODER output with NO dense layer -- the only variant the pretrained
#      inverter accepts (the standard sentence-transformers version gave BLEU 0.3 on 23 Oct).
from transformers import AutoModel
import vec2text

BGE_PREFIX = "Represent this sentence for searching relevant passages: "

@torch.no_grad()
def embed_bge(texts, model, tok, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i+bs], padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        h = model(**b).last_hidden_state[:, 0]
        out.append(torch.nn.functional.normalize(h.float(), dim=-1).cpu())
    return torch.cat(out).numpy().astype(np.float32)

@torch.no_grad()
def embed_gtr(texts, encoder, tok, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i+bs], return_tensors="pt", max_length=128, truncation=True,
                padding="max_length").to(DEVICE)
        h = encoder(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
        out.append(vec2text.models.model_utils.mean_pool(h, b["attention_mask"]).float().cpu())
    return torch.cat(out).numpy().astype(np.float32)

def cached(name, fn):
    path = f"{OUT}/{name}.npy"
    if os.path.exists(path):
        print("loaded", path); return np.load(path)
    t = time.time(); arr = fn(); np.save(path, arr)
    print(f"{name}: {arr.shape} in {time.time()-t:.0f}s"); return arr

In [ ]:
# Cell 5 - victim side: BGE embeddings of the victim DB and its questions,
# plus BGE embeddings of the attacker's passages (attacker can run any public model).
bge_tok = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")
bge = AutoModel.from_pretrained("BAAI/bge-large-en-v1.5", torch_dtype=torch.float16).to(DEVICE).eval()

X_db  = cached("bge_victim_p", lambda: embed_bge(victim_p, bge, bge_tok))
Q_db  = cached("bge_victim_q", lambda: embed_bge([BGE_PREFIX + q for q in victim_q], bge, bge_tok))
X_att = cached("bge_att_p",    lambda: embed_bge(att_p, bge, bge_tok))

del bge; torch.cuda.empty_cache()

In [ ]:
# Cell 6 - attacker side: GTR embeddings.
# Y_att: attacker's own passages (used to fit the alignment).
# Y_db : victim passages in GTR space -- ONLY used for scoring, the attacker never sees these.
gtr_enc = AutoModel.from_pretrained("sentence-transformers/gtr-t5-base").encoder.to(DEVICE).eval()
Y_att = cached("gtr_att_p",    lambda: embed_gtr(att_p, gtr_enc, gtr_tok))
Y_db  = cached("gtr_victim_p", lambda: embed_gtr(victim_p, gtr_enc, gtr_tok))
del gtr_enc; torch.cuda.empty_cache()
print("GTR norm (mean):", float(np.linalg.norm(Y_db, axis=1).mean()))

In [ ]:
# ---- Compression schemes (codecs) ----
# Each codec has three jobs:
#   fit(X)      learn what it needs from the victim database (e.g. PQ codebooks, SVD basis)
#   encode(X)   turn float vectors into what is actually stored (the compressed codes)
#   decode(C)   turn stored codes back into float vectors -- this is what an attacker
#               gets after stealing the index (the codebooks are stored inside the index)
#   search(Q,k) how a real vector DB with this compression answers a query
import numpy as np
import faiss

class Codec:
    name = "base"
    def fit(self, X): return self
    def encode(self, X): raise NotImplementedError
    def decode(self, C): raise NotImplementedError
    def bytes_per_vector(self): raise NotImplementedError
    def build(self, X):
        """Fit on the victim DB, store its compressed codes."""
        self.fit(X)
        self.codes = self.encode(X)
        return self
    def search(self, Q, k):
        raise NotImplementedError


class FP32(Codec):
    """Control: no compression."""
    name = "fp32"
    def fit(self, X):
        self.d = X.shape[1]; return self
    def encode(self, X): return np.ascontiguousarray(X, dtype=np.float32)
    def decode(self, C): return np.asarray(C, dtype=np.float32)
    def bytes_per_vector(self): return 4 * self.d
    def search(self, Q, k):
        index = faiss.IndexFlatIP(self.d)
        index.add(self.codes)
        return index.search(np.ascontiguousarray(Q, dtype=np.float32), k)[1]


class FaissCodec(Codec):
    """int8 / int4 scalar quantisation and product quantisation, all through FAISS.
    Encoding and decoding use the SAME trained index (sa_encode / sa_decode),
    so decode always matches how the DB stored the vector."""
    def __init__(self, name, factory):
        self.name, self.factory = name, factory
    def fit(self, X):
        self.d = X.shape[1]
        self.index = faiss.index_factory(self.d, self.factory, faiss.METRIC_INNER_PRODUCT)
        self.index.train(np.ascontiguousarray(X, dtype=np.float32))
        return self
    def build(self, X):
        self.fit(X)
        self.index.add(np.ascontiguousarray(X, dtype=np.float32))
        self.codes = self.encode(X)
        return self
    def encode(self, X): return self.index.sa_encode(np.ascontiguousarray(X, dtype=np.float32))
    def decode(self, C): return self.index.sa_decode(C)
    def bytes_per_vector(self): return self.index.sa_code_size()
    def search(self, Q, k):
        # Asymmetric search: the query stays float, the DB is compressed.
        # This is what a real FAISS SQ/PQ index does.
        return self.index.search(np.ascontiguousarray(Q, dtype=np.float32), k)[1]


class Binary(Codec):
    """1 bit per dimension: keep only the sign."""
    name = "binary"
    def fit(self, X):
        self.d = X.shape[1]; return self
    def encode(self, X): return np.packbits((X > 0).astype(np.uint8), axis=1)
    def decode(self, C):
        bits = np.unpackbits(C, axis=1)[:, :self.d].astype(np.float32)
        return (2.0 * bits - 1.0) / np.sqrt(self.d)   # +-1, scaled to unit length
    def bytes_per_vector(self): return self.d // 8
    def search(self, Q, k):
        # Hamming search: queries are binarised the same way as the DB.
        index = faiss.IndexBinaryFlat(self.d)
        index.add(self.codes)
        return index.search(self.encode(Q), k)[1]


class SVD(Codec):
    """Keep only the top-r directions of the victim DB (dimensionality reduction)."""
    def __init__(self, r=256):
        self.r = r; self.name = f"svd{r}"
    def fit(self, X):
        self.d = X.shape[1]
        _, _, Vt = np.linalg.svd(X.astype(np.float64), full_matrices=False)
        self.V = Vt[:self.r].T.astype(np.float32)     # d x r
        return self
    def encode(self, X): return np.ascontiguousarray(X @ self.V, dtype=np.float32)
    def decode(self, C): return np.asarray(C, dtype=np.float32)   # attacker sees the r-dim vectors as they are
    def bytes_per_vector(self): return 4 * self.r
    def search(self, Q, k):
        # Queries must go through the same projection, or the dimensions do not match.
        index = faiss.IndexFlatIP(self.r)
        index.add(self.codes)
        return index.search(self.encode(Q), k)[1]


def make_codecs():
    return [
        FP32(),
        FaissCodec("int8", "SQ8"),
        FaissCodec("int4", "SQ4"),
        FaissCodec("pq128x8", "PQ128x8"),   # 128 sub-vectors of 8 dims, 8 bits each
        Binary(),
        SVD(256),
    ]


def roundtrip_check(codec, X):
    """Correctness tests for one codec. Raises if something is wrong."""
    C = codec.encode(X)
    Xr = codec.decode(C)
    out = {"scheme": codec.name, "bytes_per_vector": int(codec.bytes_per_vector())}
    if Xr.shape[1] == X.shape[1]:
        out["rel_error"] = float(np.linalg.norm(X - Xr) / np.linalg.norm(X))
        out["mean_cos_to_original"] = float(np.mean(
            np.sum(X * Xr, 1) / (np.linalg.norm(X, axis=1) * np.linalg.norm(Xr, axis=1) + 1e-12)))
    else:
        out["rel_error"] = float("nan"); out["mean_cos_to_original"] = float("nan")
    # 1) stored code size is what we claim
    assert C.nbytes == len(X) * codec.bytes_per_vector(), (codec.name, C.nbytes)
    # 2) re-encoding a decoded vector gives the same code (decode really inverts encode)
    if codec.name not in ("fp32", "svd256"):
        C2 = codec.encode(Xr)
        out["reencode_match"] = float(np.mean(np.all(C2 == C, axis=1)))
        assert out["reencode_match"] > 0.99, (codec.name, out["reencode_match"])
    # 3) the index's own search agrees with brute force on decoded vectors
    q = X[:200]
    top_index = codec.search(q, 1)[:, 0]
    if codec.name == "binary":
        top_brute = np.argmax(codec.decode(codec.encode(q)) @ codec.decode(codec.codes).T, axis=1)
    elif codec.name.startswith("svd"):
        top_brute = np.argmax(codec.encode(q) @ codec.decode(codec.codes).T, axis=1)
    else:
        top_brute = np.argmax(q @ codec.decode(codec.codes).T, axis=1)
    out["search_agrees"] = float(np.mean(top_index == top_brute))
    assert out["search_agrees"] > 0.95, (codec.name, out["search_agrees"])
    return out


def topk_recall(codec, Q, gold, k=10):
    I = codec.search(Q, k)
    return float(np.mean([gold[i] in I[i] for i in range(len(Q))]))


# ---- Supervised linear oracle (upper bound for any linear alignment) ----
def fit_ridge(X, Y, lam):
    """Closed-form ridge regression with bias: Y ~ X W + b."""
    X = X.astype(np.float64); Y = Y.astype(np.float64)
    mx, my = X.mean(0), Y.mean(0)
    Xc, Yc = X - mx, Y - my
    W = np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ Yc)
    b = my - mx @ W
    return W.astype(np.float32), b.astype(np.float32)


def cos_rows(A, B):
    return np.sum(A * B, 1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12)


def fit_oracle(X_att, Y_att, lams=(0.1, 1.0, 10.0, 100.0), n_val=1000):
    """Pick the ridge strength on a held-out slice of the attacker's OWN data."""
    Xtr, Ytr, Xva, Yva = X_att[:-n_val], Y_att[:-n_val], X_att[-n_val:], Y_att[-n_val:]
    best = None
    for lam in lams:
        W, b = fit_ridge(Xtr, Ytr, lam)
        c = float(np.mean(cos_rows(Xva @ W + b, Yva)))
        if best is None or c > best[0]:
            best = (c, lam)
    W, b = fit_ridge(X_att, Y_att, best[1])
    return W, b, best[1]


def alignment_scores(Y_hat, Y_true):
    """top-1: is the translated vector closest to its own true GTR vector
    among all candidates in the pool? (pool = all rows of Y_true)"""
    A = Y_hat / np.linalg.norm(Y_hat, axis=1, keepdims=True)
    B = Y_true / np.linalg.norm(Y_true, axis=1, keepdims=True)
    S = A @ B.T
    top1 = float(np.mean(np.argmax(S, 1) == np.arange(len(A))))
    ranks = (S > S[np.arange(len(A)), np.arange(len(A))][:, None]).sum(1)
    return {"align_top1": top1, "align_mean_rank": float(ranks.mean() + 1),
            "align_mean_cos": float(np.mean(np.diag(S)))}

In [ ]:
# ---- Text metrics ----
import re
import sacrebleu

def _tok(s):
    return re.findall(r"\w+", s.lower())

def token_f1(pred, ref):
    p, r = _tok(pred), _tok(ref)
    if not p or not r:
        return 0.0
    rc = {}
    for w in r:
        rc[w] = rc.get(w, 0) + 1
    overlap = 0
    for w in p:
        if rc.get(w, 0) > 0:
            overlap += 1; rc[w] -= 1
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(r)
    return 2 * prec * rec / (prec + rec)

def text_scores(preds, refs):
    return {
        "bleu": float(sacrebleu.corpus_bleu(preds, [refs]).score),   # 0-100
        "token_f1": float(np.mean([token_f1(p, r) for p, r in zip(preds, refs)])) * 100,
        "exact_match": float(np.mean([p.strip() == r.strip() for p, r in zip(preds, refs)])) * 100,
    }

In [ ]:
# Cell 9 - build the 6 compressed victim databases and test each one.
# The run stops here if any codec fails a test.
import pandas as pd
codecs = make_codecs()
checks = []
for c in codecs:
    t = time.time()
    c.build(X_db)
    r = roundtrip_check(c, X_db[:2000])
    r["build_s"] = round(time.time() - t, 1)
    checks.append(r)
checks = pd.DataFrame(checks)
checks.to_csv(f"{OUT}/codec_checks.csv", index=False)
checks

In [ ]:
# Cell 10 - retrieval: does the right passage come back in the top 10 for its question?
gold = np.arange(N_VICTIM)
retrieval = {c.name: topk_recall(c, Q_db, gold, k=10) * 100 for c in codecs}
pd.Series(retrieval, name="top10_recall_%").round(1)

In [ ]:
# Cell 11 - positive control: invert the TRUE GTR vectors of the 100 test passages.
# If this is not above 80 BLEU, something is broken (wrong embedding variant, wrong
# library version) and every later number would be meaningless -- so we stop.
corrector = vec2text.load_pretrained_corrector("gtr-base")
refs = [victim_p[i] for i in inv_idx]

def invert(vectors):
    preds = []
    for i in range(0, len(vectors), INV_BATCH):
        e = torch.tensor(vectors[i:i+INV_BATCH], dtype=torch.float32, device=DEVICE)
        preds += vec2text.invert_embeddings(embeddings=e, corrector=corrector, num_steps=INV_STEPS)
    return preds

t = time.time()
ctrl_preds = invert(Y_db[inv_idx])
ctrl = text_scores(ctrl_preds, refs)
print(f"positive control: {ctrl}  ({time.time()-t:.0f}s)")
assert ctrl["bleu"] > 80, "Positive control failed. Do not trust anything below. Send this output to the team."

In [ ]:
# Cell 12 - alignment + inversion for each compressed DB.
# Attacker's view: he stole the compressed index, so he can decode it with its own codebooks,
# and he can push his OWN passages through the same codec. He fits a linear map
# (compressed BGE -> GTR) on his own pairs, then applies it to the victim's stolen vectors.
# This is a SUPERVISED upper bound: a real unsupervised attacker can only do worse or equal.
rows, examples = [], {}
pool = np.arange(1000)   # alignment top-1 is measured against 1,000 victim candidates
for c in codecs:
    t = time.time()
    X_att_c = c.decode(c.encode(X_att))          # attacker's data, compressed the same way
    W, b, lam = fit_oracle(X_att_c, Y_att)
    X_vic_c = c.decode(c.codes)                  # victim's stolen vectors
    Y_hat = X_vic_c @ W + b
    al = alignment_scores(Y_hat[pool], Y_db[pool])
    preds = invert(Y_hat[inv_idx])
    ts = text_scores(preds, refs)
    rows.append({"scheme": c.name, "bytes_per_vector": c.bytes_per_vector(),
                 "top10_recall_%": retrieval[c.name], "ridge_lambda": lam,
                 "align_top1_%": al["align_top1"] * 100, "align_mean_rank": al["align_mean_rank"],
                 "align_mean_cos": al["align_mean_cos"], **ts, "time_s": round(time.time() - t)})
    examples[c.name] = list(zip(refs[:3], preds[:3]))
    print(rows[-1])

res = pd.DataFrame(rows)
res.to_csv(f"{OUT}/results_oracle.csv", index=False)
json.dump({"positive_control": ctrl, "examples": examples, "results": rows},
          open(f"{OUT}/results_oracle.json", "w"), indent=1)
res.round(2)

In [ ]:
# Cell 13 - plot: privacy leak (BLEU) and usefulness (Top-10 retrieval) against storage size.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(res["bytes_per_vector"], res["top10_recall_%"], "o", color="tab:blue", label="Top-10 retrieval (%)")
ax.plot(res["bytes_per_vector"], res["bleu"], "s", color="tab:red", label="Reconstruction BLEU (oracle alignment)")
for _, r in res.iterrows():
    ax.annotate(r["scheme"], (r["bytes_per_vector"], r["bleu"]), textcoords="offset points", xytext=(5, 5), fontsize=9)
ax.axhline(ctrl["bleu"], color="gray", ls="--", lw=1, label=f"Positive control BLEU ({ctrl['bleu']:.1f})")
ax.set_xscale("log")
ax.set_xlabel("Storage per vector (bytes, log scale)")
ax.set_ylabel("Score (0-100)")
ax.set_title("Notebook 1: retrieval vs reconstruction per compression scheme\n(BGE-large victim, 10k NQ passages, supervised linear alignment)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/nb1_tradeoff.png", dpi=150); plt.show()

In [ ]:
# Cell 14 - read three reconstructions per scheme with your own eyes.
for name, ex in examples.items():
    print(f"=== {name}")
    for ref, pred in ex:
        print("  TRUE:", ref); print("  GOT :", pred); print()
print("Files saved in", OUT, ":", os.listdir(OUT))

## What to send back to the team after it finishes

- The table from Cell 12 (also saved as `nb1/results_oracle.csv`).
- The plot `nb1/nb1_tradeoff.png`.
- The positive-control BLEU from Cell 11.

How to read it:
- If an int8 or int4 BLEU is close to the positive control, those compressions do not protect anything, even against a linear map.
- If PQ or binary BLEU is near zero even here, then the unsupervised attack (Notebook 2) will also fail on them, because it cannot beat this supervised upper bound.
- To keep the files, click Save Version (Save & Run All) so /kaggle/working is stored with the notebook.